# GA4 RFM Segmentation & Audience Creation

Build **Recency / Frequency / Monetary (RFM)** user segments from Google Analytics 4,
cluster users with unsupervised learning, label segments from their RFM profiles,
and prepare (optionally create) GA4 audiences.

> **Before you run:** edit the **Configuration** cell. Set `USE_DEMO_DATA = False`
> and fill in your GA4 property ID + service-account path to use live data.
> Demo mode runs end-to-end with synthetic users (safe for cloning this repo).

## 1. Configuration

All environment-specific values live here. Change these constants — do not hardcode
IDs or paths elsewhere in the notebook.

In [ ]:
from pathlib import Path

# ---------------------------------------------------------------------------
# GA4 connection (required when USE_DEMO_DATA = False)
# ---------------------------------------------------------------------------
# Numeric GA4 property ID from Admin → Property Settings (digits only).
GA4_PROPERTY_ID = "YOUR_GA4_PROPERTY_ID"

# Path to a service-account JSON key with Analytics Data API access.
# Create a key in GCP IAM, then grant the SA "Viewer" (or higher) on the GA4 property.
SERVICE_ACCOUNT_PATH = Path("service_account.json")

# OAuth scopes used by the Analytics Data / Admin clients.
GA4_SCOPES = ("https://www.googleapis.com/auth/analytics.readonly",)

# ---------------------------------------------------------------------------
# Report window & identity
# ---------------------------------------------------------------------------
START_DATE = "90daysAgo"  # GA4 relative date, or "YYYY-MM-DD"
END_DATE = "yesterday"

# Dimension that uniquely identifies a user for RFM / audiences.
# Common options: "userId", or a custom user-scoped dimension such as
# "customUser:client_id" (must match your GA4 custom-dimension API name).
USER_ID_DIMENSION = "customUser:custom_client_id"

# ---------------------------------------------------------------------------
# Runtime mode
# ---------------------------------------------------------------------------
# True  → generate synthetic RFM rows (no GA4 credentials needed).
# False → pull a report from the GA4 Data API using the constants above.
USE_DEMO_DATA = True

DEMO_NUM_USERS = 5_000
RANDOM_STATE = 42

# ---------------------------------------------------------------------------
# Clustering
# ---------------------------------------------------------------------------
# Inclusive search range for KMeans; best k is chosen by silhouette score.
N_CLUSTERS_MIN = 3
N_CLUSTERS_MAX = 8

# Primary algorithm used for segment naming / audience export.
PRIMARY_CLUSTER_METHOD = "kmeans"  # "kmeans" | "agglomerative" | "dbscan"

# DBSCAN hyperparameters (used only for comparison).
DBSCAN_EPS = 0.5
DBSCAN_MIN_SAMPLES = 10

# ---------------------------------------------------------------------------
# Audience export
# ---------------------------------------------------------------------------
# Segment name to export / optionally create as a GA4 audience.
TARGET_SEGMENT_NAME = "Champions"

# If True, call the Analytics Admin API to create an audience definition.
# Requires write access on the property. Default False for safety.
CREATE_GA4_AUDIENCE = False

# Audience membership duration in days (GA4 Admin API).
AUDIENCE_MEMBERSHIP_DURATION_DAYS = 30

# ---------------------------------------------------------------------------
# Outputs
# ---------------------------------------------------------------------------
OUTPUT_DIR = Path("output")
RFM_CSV_PATH = OUTPUT_DIR / "rfm_dataset.csv"
SEGMENTS_CSV_PATH = OUTPUT_DIR / "rfm_segments.csv"

## 2. Setup

Install dependencies once (`pip install -r requirements.txt`), then run the cells below.

In [ ]:
from __future__ import annotations

import json
import warnings
from typing import Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
from IPython.display import display
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"USE_DEMO_DATA = {USE_DEMO_DATA}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

## 3. Authenticate & fetch GA4 data

When `USE_DEMO_DATA` is `False`, this section builds a Data API client from
`SERVICE_ACCOUNT_PATH` and runs an RFM-oriented report.

**Service account checklist**
1. Enable **Google Analytics Data API** (and Admin API if creating audiences) in GCP.
2. Create a service account + JSON key; store it locally (never commit it).
3. In GA4 Admin → Property access management, add the SA email with at least Viewer.

In [ ]:
def get_ga4_data_client(credentials_path: Path):
    """Return a BetaAnalyticsDataClient authenticated via service account."""
    from google.analytics.data_v1beta import BetaAnalyticsDataClient
    from google.oauth2 import service_account

    if not credentials_path.exists():
        raise FileNotFoundError(
            f"Service account file not found: {credentials_path.resolve()}. "
            "Place your JSON key there or update SERVICE_ACCOUNT_PATH."
        )

    credentials = service_account.Credentials.from_service_account_file(
        str(credentials_path),
        scopes=list(GA4_SCOPES),
    )
    return BetaAnalyticsDataClient(credentials=credentials)


def run_ga4_rfm_report(client, property_id: str) -> pd.DataFrame:
    """Pull user × date activity for RFM aggregation."""
    from google.analytics.data_v1beta.types import (
        DateRange,
        Dimension,
        Filter,
        FilterExpression,
        Metric,
        NumericValue,
        RunReportRequest,
    )

    if not property_id or property_id.startswith("YOUR_"):
        raise ValueError(
            "Set GA4_PROPERTY_ID to your numeric property ID before pulling live data."
        )

    request = RunReportRequest(
        property=f"properties/{property_id}",
        date_ranges=[DateRange(start_date=START_DATE, end_date=END_DATE)],
        dimensions=[
            Dimension(name=USER_ID_DIMENSION),
            Dimension(name="date"),
        ],
        metrics=[
            Metric(name="sessions"),
            Metric(name="transactions"),
            Metric(name="totalRevenue"),
        ],
        metric_filter=FilterExpression(
            filter=Filter(
                field_name="totalRevenue",
                numeric_filter=Filter.NumericFilter(
                    operation=Filter.NumericFilter.Operation.GREATER_THAN,
                    value=NumericValue(double_value=0.0),
                ),
            )
        ),
        limit=100_000,
    )

    response = client.run_report(request)
    if not response.rows:
        raise RuntimeError(
            "GA4 returned 0 rows. Check the date range, property ID, "
            "USER_ID_DIMENSION, and that purchases exist in the window."
        )

    records = []
    for row in response.rows:
        records.append(
            {
                "user_id": row.dimension_values[0].value,
                "date": row.dimension_values[1].value,
                "sessions": float(row.metric_values[0].value),
                "transactions": float(row.metric_values[1].value),
                "total_revenue": float(row.metric_values[2].value),
            }
        )

    raw = pd.DataFrame(records)
    raw = raw[raw["user_id"].ne("(not set)") & raw["user_id"].ne("")]
    raw["date"] = pd.to_datetime(raw["date"], format="%Y%m%d", errors="coerce")
    return raw


def aggregate_rfm(raw: pd.DataFrame, as_of: pd.Timestamp | None = None) -> pd.DataFrame:
    """Aggregate event-level rows into one RFM row per user."""
    as_of = as_of or pd.Timestamp.today().normalize()
    last_dates = raw.groupby("user_id", as_index=False)["date"].max()
    last_dates = last_dates.rename(columns={"date": "last_activity_date"})

    agg = (
        raw.groupby("user_id", as_index=False)
        .agg(
            frequency=("sessions", "sum"),
            transactions=("transactions", "sum"),
            monetary=("total_revenue", "sum"),
        )
    )
    rfm = agg.merge(last_dates, on="user_id", how="left")
    rfm["recency"] = (as_of - rfm["last_activity_date"]).dt.days.clip(lower=0)
    return rfm[["user_id", "recency", "frequency", "transactions", "monetary"]].copy()

In [ ]:
def make_demo_rfm(n_users: int, random_state: int = 42) -> pd.DataFrame:
    """Synthetic RFM table for offline demos (no real users)."""
    rng = np.random.default_rng(random_state)

    # Mixture of loose archetypes so clustering has structure to find.
    weights = np.array([0.15, 0.20, 0.20, 0.15, 0.15, 0.15])
    labels = rng.choice(len(weights), size=n_users, p=weights)

    # (recency_mean, frequency_mean, monetary_mean)
    centers = [
        (5, 25, 600),    # Champions
        (12, 12, 250),   # Loyal
        (8, 4, 80),      # Promising / new
        (40, 10, 300),   # At risk
        (70, 15, 500),   # Can't lose
        (80, 2, 40),     # Hibernating / lost
    ]

    rows = []
    for i, lab in enumerate(labels):
        r_mu, f_mu, m_mu = centers[lab]
        rows.append(
            {
                "user_id": f"demo_user_{i+1:05d}",
                "recency": int(np.clip(rng.normal(r_mu, r_mu * 0.35), 1, 120)),
                "frequency": int(np.clip(rng.normal(f_mu, max(f_mu * 0.4, 1)), 1, 80)),
                "transactions": int(np.clip(rng.normal(max(f_mu / 3, 1), 2), 1, 40)),
                "monetary": float(np.clip(rng.normal(m_mu, m_mu * 0.45), 5, 3000)),
            }
        )
    return pd.DataFrame(rows)


if USE_DEMO_DATA:
    rfm_data = make_demo_rfm(DEMO_NUM_USERS, RANDOM_STATE)
    print(f"Loaded demo RFM data: {len(rfm_data):,} users")
else:
    client = get_ga4_data_client(SERVICE_ACCOUNT_PATH)
    raw_events = run_ga4_rfm_report(client, GA4_PROPERTY_ID)
    rfm_data = aggregate_rfm(raw_events)
    print(f"Loaded GA4 RFM data: {len(rfm_data):,} users")

rfm_data.to_csv(RFM_CSV_PATH, index=False)
print(f"Wrote {RFM_CSV_PATH}")
rfm_data.head()

## 4. Exploratory data analysis

Inspect distributions, correlations, and outliers before clustering.
Missing values and duplicates are checked explicitly.

In [ ]:
FEATURE_COLS = ["recency", "frequency", "monetary"]

print("Shape:", rfm_data.shape)
print("\nMissing values:\n", rfm_data[FEATURE_COLS].isna().sum())
print("\nDuplicate user_ids:", rfm_data["user_id"].duplicated().sum())
display(rfm_data[FEATURE_COLS].describe().T)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
colors = ["#4C78A8", "#54A24B", "#E45756"]

for ax, col, color in zip(axes, FEATURE_COLS, colors):
    sns.histplot(rfm_data[col], bins=30, ax=ax, color=color, kde=True)
    ax.set_title(f"{col.title()} distribution")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(
    rfm_data[FEATURE_COLS].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    ax=axes[0],
)
axes[0].set_title("RFM correlation")

sns.boxplot(data=rfm_data[FEATURE_COLS], orient="h", ax=axes[1], palette=colors)
axes[1].set_title("RFM outliers (box plots)")

plt.tight_layout()
plt.show()

## 5. Feature scaling & cluster selection

Scale RFM features, score KMeans across `N_CLUSTERS_MIN`…`N_CLUSTERS_MAX` with
silhouette score, then fit comparison models (Agglomerative, DBSCAN).

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(rfm_data[FEATURE_COLS])

cluster_range = range(N_CLUSTERS_MIN, N_CLUSTERS_MAX + 1)
silhouette_scores = []

for k in cluster_range:
    labels = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit_predict(X_scaled)
    silhouette_scores.append(silhouette_score(X_scaled, labels))

best_k = list(cluster_range)[int(np.argmax(silhouette_scores))]
print(f"Best k by silhouette: {best_k} (score={max(silhouette_scores):.3f})")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(cluster_range), silhouette_scores, marker="o", color="#4C78A8")
ax.axvline(best_k, color="#E45756", linestyle="--", label=f"best k={best_k}")
ax.set_xlabel("Number of clusters (k)")
ax.set_ylabel("Silhouette score")
ax.set_title("KMeans model selection")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
rfm_data["kmeans_cluster"] = kmeans.fit_predict(X_scaled)

agglo = AgglomerativeClustering(n_clusters=best_k)
rfm_data["agglomerative_cluster"] = agglo.fit_predict(X_scaled)

dbscan = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES)
rfm_data["dbscan_cluster"] = dbscan.fit_predict(X_scaled)

print("KMeans sizes:\n", rfm_data["kmeans_cluster"].value_counts().sort_index())
print("\nAgglomerative sizes:\n", rfm_data["agglomerative_cluster"].value_counts().sort_index())
print("\nDBSCAN sizes (-1 = noise):\n", rfm_data["dbscan_cluster"].value_counts().sort_index())

## 6. Visualize clusters (PCA)

Project scaled RFM features to 2D with PCA for an interpretable scatter plot.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(X_scaled)
rfm_data["pca_1"] = coords[:, 0]
rfm_data["pca_2"] = coords[:, 1]
print(
    f"PCA explained variance: "
    f"{pca.explained_variance_ratio_[0]:.1%} + {pca.explained_variance_ratio_[1]:.1%} "
    f"= {pca.explained_variance_ratio_.sum():.1%}"
)

fig = px.scatter(
    rfm_data,
    x="pca_1",
    y="pca_2",
    color=rfm_data["kmeans_cluster"].astype(str),
    hover_data=["user_id", "recency", "frequency", "monetary"],
    title="KMeans clusters in PCA space",
    labels={"color": "cluster"},
)
fig.show()

In [ ]:
# Optional interactive 3D view of original RFM features
fig_3d = px.scatter_3d(
    rfm_data,
    x="recency",
    y="frequency",
    z="monetary",
    color=rfm_data["kmeans_cluster"].astype(str),
    hover_data=["user_id"],
    title="KMeans clusters in RFM space",
    labels={"color": "cluster"},
)
fig_3d.show()

## 7. Profile clusters & assign business labels

Labels are derived from each cluster's mean RFM profile (not hard-coded cluster IDs),
so names stay meaningful when `k` or the data changes.

In [ ]:
SEGMENT_RULES = [
    # (name, recency_rank preference, frequency_rank preference, monetary_rank preference)
    # Ranks: "low" / "mid" / "high" relative to other clusters.
    ("Champions", "low", "high", "high"),
    ("Loyal Customers", "low", "high", "mid"),
    ("Potential Loyalists", "low", "mid", "mid"),
    ("Recent Customers", "low", "low", "low"),
    ("Need Attention", "mid", "mid", "mid"),
    ("At Risk", "high", "high", "high"),
    ("Can't Lose Them", "high", "high", "mid"),
    ("Hibernating", "high", "low", "low"),
]


def _rank_bucket(series: pd.Series) -> pd.Series:
    """Map cluster means to low/mid/high tertiles."""
    if series.nunique() == 1:
        return pd.Series("mid", index=series.index)
    # Higher raw value → higher rank; for recency, high means worse (stale).
    try:
        return pd.qcut(series.rank(method="first"), q=3, labels=["low", "mid", "high"])
    except ValueError:
        return pd.Series("mid", index=series.index)


def label_clusters_from_profiles(df: pd.DataFrame, cluster_col: str) -> dict[int, str]:
    """Return mapping cluster_id → segment name based on mean RFM."""
    profile = (
        df.groupby(cluster_col)[FEATURE_COLS]
        .mean()
        .rename(columns=lambda c: f"mean_{c}")
    )
    # Drop noise label from DBSCAN if present
    profile = profile[profile.index != -1]

    if profile.empty:
        return {}

    r_bucket = _rank_bucket(profile["mean_recency"])
    f_bucket = _rank_bucket(profile["mean_frequency"])
    m_bucket = _rank_bucket(profile["mean_monetary"])

    used_names: set[str] = set()
    mapping: dict[int, str] = {}

    for cluster_id in profile.index:
        key = (str(r_bucket.loc[cluster_id]), str(f_bucket.loc[cluster_id]), str(m_bucket.loc[cluster_id]))
        name = next(
            (n for n, r, f, m in SEGMENT_RULES if (r, f, m) == key and n not in used_names),
            None,
        )
        if name is None:
            # Fallback: score = F + M - R (standardized within profile table)
            z = profile.apply(lambda s: (s - s.mean()) / (s.std() or 1.0))
            score = (
                -z.loc[cluster_id, "mean_recency"]
                + z.loc[cluster_id, "mean_frequency"]
                + z.loc[cluster_id, "mean_monetary"]
            )
            name = f"Segment {cluster_id} (score={score:.2f})"
        used_names.add(name)
        mapping[int(cluster_id)] = name

    return mapping


def apply_segment_names(df: pd.DataFrame, methods: Iterable[str]) -> pd.DataFrame:
    out = df.copy()
    for method in methods:
        col = f"{method}_cluster"
        name_col = f"{method}_segment"
        mapping = label_clusters_from_profiles(out, col)
        out[name_col] = out[col].map(mapping)
        out.loc[out[col] == -1, name_col] = "Noise / Outlier"
    return out


rfm_data = apply_segment_names(
    rfm_data, methods=["kmeans", "agglomerative", "dbscan"]
)

primary_segment_col = f"{PRIMARY_CLUSTER_METHOD}_segment"
profile_table = (
    rfm_data.groupby(primary_segment_col)[FEATURE_COLS]
    .agg(["count", "mean", "median"])
)
display(profile_table)

In [ ]:
segment_counts = rfm_data[primary_segment_col].value_counts().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, max(4, 0.4 * len(segment_counts))))
segment_counts.plot(kind="barh", ax=ax, color="#4C78A8")
ax.set_title(f"Users per segment ({PRIMARY_CLUSTER_METHOD})")
ax.set_xlabel("Users")
plt.tight_layout()
plt.show()

fig = px.scatter(
    rfm_data,
    x="pca_1",
    y="pca_2",
    color=primary_segment_col,
    hover_data=["user_id", "recency", "frequency", "monetary"],
    title="Named segments in PCA space",
)
fig.show()

## 8. Export segments & prepare GA4 audiences

Writes `output/rfm_segments.csv` and builds the user-ID list for `TARGET_SEGMENT_NAME`.

Audience creation via the Admin API is **opt-in** (`CREATE_GA4_AUDIENCE = True`)
and requires the service account to have Editor (or equivalent) on the property.

In [ ]:
export_cols = [
    "user_id",
    "recency",
    "frequency",
    "transactions",
    "monetary",
    "kmeans_cluster",
    "kmeans_segment",
    "agglomerative_cluster",
    "agglomerative_segment",
    "dbscan_cluster",
    "dbscan_segment",
]
rfm_data[export_cols].to_csv(SEGMENTS_CSV_PATH, index=False)
print(f"Wrote {SEGMENTS_CSV_PATH}")

audience_user_ids = (
    rfm_data.loc[rfm_data[primary_segment_col] == TARGET_SEGMENT_NAME, "user_id"]
    .astype(str)
    .tolist()
)
print(f"Segment '{TARGET_SEGMENT_NAME}': {len(audience_user_ids):,} users")
if audience_user_ids:
    print("Sample IDs:", audience_user_ids[:5])
else:
    print(
        f"No users matched TARGET_SEGMENT_NAME={TARGET_SEGMENT_NAME!r}. "
        f"Available: {sorted(rfm_data[primary_segment_col].dropna().unique())}"
    )

In [ ]:
def build_audience_template(
    property_id: str,
    audience_name: str,
    user_ids: list[str],
    user_id_dimension: str,
    membership_days: int,
) -> dict:
    """Return a documentation-friendly GA4 audience filter template.

    Large ID lists are better activated via CSV export + Google Ads Customer Match,
    CDP sync, or the GA4 UI. Embedding thousands of IDs in an Admin API audience
    definition is brittle and subject to size limits.
    """
    return {
        "parent": f"properties/{property_id}",
        "audience": {
            "displayName": audience_name,
            "description": f"RFM segment '{audience_name}' from clustering notebook.",
            "membershipDurationDays": membership_days,
            "filterClauses": [
                {
                    "clauseType": "INCLUDE",
                    "simpleFilter": {
                        "scope": "AUDIENCE_FILTER_SCOPE_ACROSS_USERS",
                        "filterExpression": {
                            "andGroup": {
                                "filterExpressions": [
                                    {
                                        "dimensionOrMetricFilter": {
                                            "fieldName": user_id_dimension,
                                            "inListFilter": {
                                                "values": user_ids,
                                                "caseSensitive": True,
                                            },
                                        }
                                    }
                                ]
                            }
                        },
                    },
                }
            ],
        },
    }


def maybe_create_ga4_audience(credentials_path: Path, payload: dict):
    """POST audience JSON to Analytics Admin API (opt-in)."""
    import requests
    from google.auth.transport.requests import Request
    from google.oauth2 import service_account

    credentials = service_account.Credentials.from_service_account_file(
        str(credentials_path),
        scopes=["https://www.googleapis.com/auth/analytics.edit"],
    )
    credentials.refresh(Request())

    url = f"https://analyticsadmin.googleapis.com/v1alpha/{payload['parent']}/audiences"
    response = requests.post(
        url,
        headers={
            "Authorization": f"Bearer {credentials.token}",
            "Content-Type": "application/json",
        },
        json=payload["audience"],
        timeout=60,
    )
    if not response.ok:
        raise RuntimeError(
            f"Audience create failed ({response.status_code}): {response.text}"
        )
    return response.json()


audience_ids_path = OUTPUT_DIR / f"audience_{TARGET_SEGMENT_NAME.replace(' ', '_').lower()}_user_ids.json"
with audience_ids_path.open("w", encoding="utf-8") as fh:
    json.dump(
        {
            "segment": TARGET_SEGMENT_NAME,
            "user_id_dimension": USER_ID_DIMENSION,
            "user_ids": audience_user_ids,
        },
        fh,
        indent=2,
    )
print(f"Wrote {audience_ids_path} ({len(audience_user_ids):,} ids)")

payload = None
if audience_user_ids:
    MAX_INLINE_IDS = 500
    inline_ids = audience_user_ids[:MAX_INLINE_IDS]
    if len(audience_user_ids) > MAX_INLINE_IDS:
        print(
            f"Note: segment has {len(audience_user_ids)} users; "
            f"template embeds only the first {MAX_INLINE_IDS}. "
            "Use the JSON/CSV export for full activation."
        )

    payload = build_audience_template(
        property_id=GA4_PROPERTY_ID,
        audience_name=TARGET_SEGMENT_NAME,
        user_ids=inline_ids,
        user_id_dimension=USER_ID_DIMENSION,
        membership_days=AUDIENCE_MEMBERSHIP_DURATION_DAYS,
    )
    print("Audience template (truncated):")
    print(json.dumps(payload, indent=2)[:1500], "...")

if CREATE_GA4_AUDIENCE:
    if USE_DEMO_DATA:
        raise RuntimeError("Refusing to create a GA4 audience while USE_DEMO_DATA=True.")
    if not payload:
        raise RuntimeError("No audience payload to create.")
    created = maybe_create_ga4_audience(SERVICE_ACCOUNT_PATH, payload)
    print("Created audience:", json.dumps(created, indent=2)[:1000])
else:
    print("CREATE_GA4_AUDIENCE is False — exported IDs / template only (no API write).")

## Next steps

After you run the notebook on your property:

1. Inspect segment sizes and mean RFM values — adjust `N_CLUSTERS_*` if segments are too coarse/fine.
2. Activate high-value segments in Google Ads / Magento / your ESP via the exported CSV.
3. Re-run on a schedule (weekly/monthly) so recency stays current.
4. Consider adding product affinity or channel features alongside RFM for richer clusters.

### Suggested validation checklist
- [ ] `GA4_PROPERTY_ID` and `USER_ID_DIMENSION` match your property
- [ ] Service account can run Data API reports
- [ ] Demo mode produces sensible segment profiles before switching to live data
- [ ] No credential files are committed (`git status` clean of `*.json` keys)